## 02. Data Cleaning & Preparation

### 1. Data Loading & Setup

In [1]:
# Import libraries and configure dataset paths

import pandas as pd
import csv
import re
from pathlib import Path
from collections import Counter

raw_path = Path("../data/raw")
processed_path = Path("../data/processed")

processed_path.mkdir(parents=True, exist_ok=True)

csv.field_size_limit(2**31 - 1)

files = {
    "Movies": "movie_data.csv",
    "Ratings": "ratings_export.csv",
    "Users": "users_export.csv"
}

for name, filename in files.items():
    file_path = raw_path / filename
    print(f"{name}: {'Found' if file_path.is_file() else 'Missing'}")


Movies: Found
Ratings: Found
Users: Found


### 2. Movies Data Cleaning

#### 2.1 CSV Structure Repair

In [2]:
# Repair fragmented movie records

source_file = raw_path / "movie_data.csv"
output_file = processed_path / "movie_data_repaired.csv"

repair_counts = Counter()
pending_row = None

with open(source_file, encoding="utf-8-sig", newline="") as source:
    with open(output_file, "w", encoding="utf-8", newline="") as output:

        reader = csv.reader(source)
        writer = csv.writer(output)

        header = next(reader)
        writer.writerow(header)

        for row in reader:

            if pending_row is None:

                if len(row) == 19:
                    writer.writerow(row)
                    repair_counts["Valid records"] += 1

                elif len(row) == 9:
                    pending_row = row

                else:
                    raise ValueError(f"Unexpected row length: {len(row)}")

            else:

                if len(row) == 1:
                    pending_row[-1] += "\n" + row[0]

                elif len(row) == 11:
                    pending_row[-1] += "\n" + row[0]

                    repaired_row = pending_row + row[1:]

                    if len(repaired_row) != 19:
                        raise ValueError("Repair failed: incorrect column count")

                    writer.writerow(repaired_row)
                    repair_counts["Repaired records"] += 1

                    pending_row = None

                else:
                    raise ValueError(f"Unexpected continuation: {len(row)}")

        if pending_row is not None:
            raise ValueError("Incomplete movie record at end of file")

print(repair_counts)
print("Total movie records:", sum(repair_counts.values()))


Counter({'Valid records': 285390, 'Repaired records': 114})
Total movie records: 285504


In [3]:
# Validate repaired CSV structure

total_records = 0
invalid_rows = 0
invalid_ids = 0
duplicate_ids = 0

seen_ids = set()

with open(output_file, encoding="utf-8", newline="") as f:
    reader = csv.reader(f)
    header = next(reader)

    id_index = header.index("_id")

    for row in reader:
        total_records += 1

        if len(row) != len(header):
            invalid_rows += 1
            continue

        movie_id = row[id_index]

        if not re.fullmatch(r"[0-9a-fA-F]{24}", movie_id):
            invalid_ids += 1

        if movie_id in seen_ids:
            duplicate_ids += 1

        seen_ids.add(movie_id)

print("Total records:", total_records)
print("Invalid row structures:", invalid_rows)
print("Invalid _id values:", invalid_ids)
print("Duplicate _id values:", duplicate_ids)


Total records: 285504
Invalid row structures: 0
Invalid _id values: 0
Duplicate _id values: 0


In [4]:
# Inspect recovered movie records

repaired_ids = set()

with open(source_file, encoding="utf-8-sig", newline="") as f:
    reader = csv.reader(f)
    next(reader)

    for row in reader:
        if len(row) == 9:
            repaired_ids.add(row[0])

movies_repaired = pd.read_csv(
    output_file,
    dtype=str,
    keep_default_na=False
)

recovered_movies = movies_repaired[
    movies_repaired["_id"].isin(repaired_ids)
]

print("Expected recovered records:", len(repaired_ids))
print("Recovered records found:", len(recovered_movies))


Expected recovered records: 114
Recovered records found: 114


In [5]:
# Validate numeric fields in recovered records

numeric_columns = [
    "popularity",
    "runtime",
    "vote_average",
    "vote_count",
    "year_released"
]

for col in numeric_columns:
    values = recovered_movies[col].replace(
        ["", "null", "none", "nan"], pd.NA
    ).dropna()

    invalid_count = pd.to_numeric(
        values, errors="coerce"
    ).isna().sum()

    print(f"{col}: {invalid_count} invalid values")


popularity: 0 invalid values
runtime: 0 invalid values
vote_average: 0 invalid values
vote_count: 0 invalid values
year_released: 0 invalid values
